# USDA-Net: Subject-Independent 3-Class Motor-Imagery EEG on PhysioNet EEGMMIDB

**Research target:** strict Leave-One-Subject-Out (LOSO) evaluation for **Left Hand / Right Hand / Feet** motor imagery.

This notebook implements the proposed **USDA-Net (Universal Subject-Domain Adaptive Network)** with:
- multi-scale temporal convolutions
- a learnable FIR filterbank
- spatial/channel attention
- Transformer sequence modeling
- gradient-reversal subject-domain adversarial learning (DANN)
- class prototype alignment
- supervised contrastive learning
- subject-wise LOSO evaluation
- leakage-safe channel-wise normalization
- publication metrics, confusion matrices, per-subject results, and confidence intervals
- optional unlabeled target adaptation as a **separate** experiment

The notebook assumes the PhysioNet EEG Motor Movement/Imagery Dataset is already downloaded locally. The official dataset documentation states that recordings are 64-channel EEG sampled at 160 Hz, with T0/T1/T2 annotations whose meanings depend on the run type. For the 3-class imagery task used here, the code maps unilateral imagery runs R04/R08/R12 to left/right fist and bilateral imagery runs R06/R10/R14 to feet. See the source note in the accompanying response.

In [1]:
# CELL 1 — Install / verify dependencies
# Run once in a fresh environment if needed.

%pip -q install mne numpy scipy pandas scikit-learn matplotlib tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
# CELL 2 — Imports and reproducibility

import os
import re
import gc
import json
import math
import random
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import scipy
from scipy import signal, stats

import mne
mne.set_log_level("WARNING")

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    cohen_kappa_score, confusion_matrix, classification_report
)

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("MNE:", mne.__version__)
print("SciPy:", scipy.__version__)
print("Device:", DEVICE)

PyTorch: 2.10.0
MNE: 1.11.0
SciPy: 1.15.3
Device: cpu


In [4]:
# CELL 3 — Research configuration

@dataclass
class Config:
    # -------- local data --------
    DATA_ROOT: str = "./cross_dataset_mi_project"   # <-- CHANGE THIS
    CACHE_DIR: str = "./physionet_3class_cache"
    OUTPUT_DIR: str = "./usda_net_results"

    # -------- dataset --------
    SFREQ: int = 160
    LOWCUT: float = 4.0
    HIGHCUT: float = 38.0
    NOTCH: float = 50.0
    TMIN: float = 0.5
    TMAX: float = 4.5
    N_CLASSES: int = 3
    CLASS_NAMES: Tuple[str, ...] = ("Left Hand", "Right Hand", "Feet")
    # PhysioNet EEGMMIDB imagery runs used for this 3-class problem.
    IMAGERY_RUNS: Tuple[int, ...] = (4, 6, 8, 10, 12, 14)

    # -------- model --------
    MODEL_CHANNELS: int = 32
    TEMP_BRANCH_CHANNELS: int = 48
    SPECTRAL_BANDS: Tuple[Tuple[float, float], ...] = (
        (4, 8), (8, 12), (12, 16), (16, 22), (22, 30), (30, 38)
    )
    FILTER_LEN: int = 63
    TOKEN_DIM: int = 128
    N_HEADS: int = 4
    N_TRANSFORMER_LAYERS: int = 3
    FF_DIM: int = 256
    DROPOUT: float = 0.15
    EMBED_DIM: int = 128
    DOMAIN_HIDDEN: int = 128

    # -------- training --------
    BATCH_SIZE: int = 64
    NUM_WORKERS: int = 0
    EPOCHS: int = 60
    WARMUP_EPOCHS: int = 8
    PATIENCE: int = 10
    LR: float = 2e-3
    WEIGHT_DECAY: float = 1e-4
    LABEL_SMOOTHING: float = 0.05
    CLIP_NORM: float = 2.0

    # Loss coefficients
    LAMBDA_DOMAIN_MAX: float = 0.7
    LAMBDA_PROTO: float = 0.05
    LAMBDA_SUPCON: float = 0.10
    SUPCON_TEMP: float = 0.10

    # -------- data augmentation --------
    NOISE_STD: float = 0.02
    AMP_SCALE_MIN: float = 0.90
    AMP_SCALE_MAX: float = 1.10
    CHANNEL_DROP_P: float = 0.08
    TIME_MASK_P: float = 0.20
    TIME_MASK_MAX_FRAC: float = 0.08
    MAX_SHIFT: int = 12

    # -------- evaluation --------
    VAL_SUBJECT_FRAC: float = 0.10
    MAX_FOLDS: Optional[int] = None   # Set e.g. 5 for development, None for all subjects.
    QUICK_FOLDS: int = 3
    RUN_QUICK_DEMO: bool = True
    RUN_FULL_LOSO: bool = False       # Set True for the full paper run.
    SAVE_CHECKPOINTS: bool = True

    # -------- optional target adaptation --------
    RUN_TARGET_ADAPTATION: bool = False
    ADAPT_EPOCHS: int = 5
    ADAPT_LR: float = 2e-4
    PSEUDO_CONFIDENCE: float = 0.90

cfg = Config()

Path(cfg.CACHE_DIR).mkdir(parents=True, exist_ok=True)
Path(cfg.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(json.dumps(asdict(cfg), indent=2, default=str))

{
  "DATA_ROOT": "./cross_dataset_mi_project",
  "CACHE_DIR": "./physionet_3class_cache",
  "OUTPUT_DIR": "./usda_net_results",
  "SFREQ": 160,
  "LOWCUT": 4.0,
  "HIGHCUT": 38.0,
  "NOTCH": 50.0,
  "TMIN": 0.5,
  "TMAX": 4.5,
  "N_CLASSES": 3,
  "CLASS_NAMES": [
    "Left Hand",
    "Right Hand",
    "Feet"
  ],
  "IMAGERY_RUNS": [
    4,
    6,
    8,
    10,
    12,
    14
  ],
  "MODEL_CHANNELS": 32,
  "TEMP_BRANCH_CHANNELS": 48,
  "SPECTRAL_BANDS": [
    [
      4,
      8
    ],
    [
      8,
      12
    ],
    [
      12,
      16
    ],
    [
      16,
      22
    ],
    [
      22,
      30
    ],
    [
      30,
      38
    ]
  ],
  "FILTER_LEN": 63,
  "TOKEN_DIM": 128,
  "N_HEADS": 4,
  "N_TRANSFORMER_LAYERS": 3,
  "FF_DIM": 256,
  "DROPOUT": 0.15,
  "EMBED_DIM": 128,
  "DOMAIN_HIDDEN": 128,
  "BATCH_SIZE": 64,
  "NUM_WORKERS": 0,
  "EPOCHS": 60,
  "WARMUP_EPOCHS": 8,
  "PATIENCE": 10,
  "LR": 0.002,
  "WEIGHT_DECAY": 0.0001,
  "LABEL_SMOOTHING": 0.05,
  "CLIP_NORM": 2.0

In [5]:
# CELL 4 — Set dataset path automatically when possible

# Common local layouts are supported:
#   root/S001/S001R04.edf
#   root/S001R04.edf
#   root/**/S001R04.edf
#   root/**/S001/**

candidate_roots = [
    Path.home() / "Downloads" / "EEGMMIDB",
    Path.home() / "Desktop" / "EEGMMIDB",
    Path.home() / "Documents" / "EEGMMIDB",
    Path("/kaggle/input/eeg-motor-movementimagery-dataset-1-0-0"),
]

if cfg.DATA_ROOT == "/path/to/EEGMMIDB":
    for p in candidate_roots:
        if p.exists() and any(p.rglob("*.edf")):
            cfg.DATA_ROOT = str(p)
            print("Auto-detected DATA_ROOT:", cfg.DATA_ROOT)
            break

print("Current DATA_ROOT:", cfg.DATA_ROOT)
print("Edit cfg.DATA_ROOT in CELL 3 if this is incorrect.")

Current DATA_ROOT: ./cross_dataset_mi_project
Edit cfg.DATA_ROOT in CELL 3 if this is incorrect.


In [6]:
# CELL 5 — Dataset discovery

def find_edf_for_run(root: Path, subject: str, run: int) -> Optional[Path]:
    target = f"{subject}R{run:02d}.edf".lower()
    # exact recursive match
    matches = [p for p in root.rglob("*.edf") if p.name.lower() == target]
    return matches[0] if matches else None


def discover_subjects(root: str) -> Dict[str, Dict[int, Path]]:
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"DATA_ROOT does not exist: {root.resolve()}")

    subject_map = {}
    for edf in root.rglob("*.edf"):
        m = re.match(r"^(S\d{3})R(\d{2})\.edf$", edf.name, re.I)
        if not m:
            continue
        subject = m.group(1).upper()
        run = int(m.group(2))
        subject_map.setdefault(subject, {})[run] = edf

    subject_map = {
        s: runs for s, runs in sorted(subject_map.items())
        if any(r in runs for r in cfg.IMAGERY_RUNS)
    }
    return subject_map

subjects_map = discover_subjects(cfg.DATA_ROOT)
subjects = sorted(subjects_map)

print(f"Discovered {len(subjects)} subjects")
print("First subjects:", subjects[:10])
if subjects:
    print("Example files:")
    for r in sorted(subjects_map[subjects[0]])[:6]:
        print(" ", r, subjects_map[subjects[0]][r])

Discovered 0 subjects
First subjects: []


In [7]:
# CELL 6 — PhysioNet run-to-class mapping

# Official annotation semantics depend on run type.
# 3-class imagery mapping used here:
#   R04, R08, R12: T1 = left fist imagery, T2 = right fist imagery
#   R06, R10, R14: T1 = both fists, T2 = BOTH FEET imagery
# We keep only the feet condition from the bilateral imagery runs.

def map_event_to_class(run: int, annotation: str) -> Optional[int]:
    annotation = annotation.upper()
    if run in (4, 8, 12):
        if annotation == "T1":
            return 0  # Left Hand
        if annotation == "T2":
            return 1  # Right Hand
    elif run in (6, 10, 14):
        if annotation == "T2":
            return 2  # Feet
    return None

for run in cfg.IMAGERY_RUNS:
    print(run, {a: map_event_to_class(run, a) for a in ["T0", "T1", "T2"]})

4 {'T0': None, 'T1': 0, 'T2': 1}
6 {'T0': None, 'T1': None, 'T2': 2}
8 {'T0': None, 'T1': 0, 'T2': 1}
10 {'T0': None, 'T1': None, 'T2': 2}
12 {'T0': None, 'T1': 0, 'T2': 1}
14 {'T0': None, 'T1': None, 'T2': 2}


In [8]:
# CELL 7 — Subject preprocessing and caching

# The preprocessing is performed once per subject and then cached as NPZ.
# We fit normalization later, inside each LOSO fold, so no test-subject statistics leak into training.

def get_eeg_channel_names(raw):
    # EDF annotation channels are not EEG. Keep channels with type 'eeg'.
    picks = mne.pick_types(raw.info, eeg=True, meg=False, eog=False, stim=False, misc=False, exclude=[])
    return picks


def preprocess_subject(subject: str, subject_map: Dict[int, Path], force_recompute: bool = False):
    cache_path = Path(cfg.CACHE_DIR) / f"{subject}_3class.npz"
    if cache_path.exists() and not force_recompute:
        d = np.load(cache_path, allow_pickle=False)
        return d["X"].astype(np.float32), d["y"].astype(np.int64)

    X_list, y_list = [], []
    sfreq_seen = None

    for run in cfg.IMAGERY_RUNS:
        edf_path = subject_map.get(run)
        if edf_path is None:
            continue

        raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
        sfreq = float(raw.info["sfreq"])
        sfreq_seen = sfreq

        # EDF EEG is expected to be 160 Hz. Resampling here makes the function robust to a local variant.
        if not np.isclose(sfreq, cfg.SFREQ):
            raw.resample(cfg.SFREQ, npad="auto", verbose=False)

        # Use EEG channels only; preserve the dataset montage.
        picks = get_eeg_channel_names(raw)
        if len(picks) == 0:
            raise RuntimeError(f"No EEG channels found in {edf_path}")
        raw.pick(picks)

        # Basic research preprocessing.
        raw.load_data()
        raw.filter(cfg.LOWCUT, cfg.HIGHCUT, method="fir", phase="zero", verbose=False)
        if cfg.NOTCH < cfg.SFREQ / 2:
            raw.notch_filter(cfg.NOTCH, method="fir", phase="zero", verbose=False)
        raw.set_eeg_reference("average", projection=False, verbose=False)

        # Obtain events from PhysioNet annotations.
        events, event_id = mne.events_from_annotations(raw, verbose=False)
        inv_event_id = {v: k for k, v in event_id.items()}

        selected = []
        labels = []
        for ev in events:
            code = inv_event_id.get(int(ev[2]), None)
            if code is None:
                continue
            cls = map_event_to_class(run, code)
            if cls is not None:
                selected.append(ev)
                labels.append(cls)

        if not selected:
            continue

        selected = np.asarray(selected, dtype=int)
        labels = np.asarray(labels, dtype=int)

        # 4 s window centered on the imagery period. T0/rest is never used.
        # We disable annotation rejection here so event/label alignment remains exact.
        # The recordings are short, cue-locked windows; later we can add explicit artifact rejection.
        epochs = mne.Epochs(
            raw,
            selected,
            event_id=None,
            tmin=cfg.TMIN,
            tmax=cfg.TMAX,
            baseline=None,
            preload=True,
            reject_by_annotation=False,
            verbose=False,
        )

        xe = epochs.get_data(copy=True).astype(np.float32)

        X_list.append(xe)
        y_list.append(labels)

        del raw, epochs, xe
        gc.collect()

    if not X_list:
        raise RuntimeError(f"No usable 3-class imagery epochs found for {subject}")

    X = np.concatenate(X_list, axis=0).astype(np.float32)
    y = np.concatenate(y_list, axis=0).astype(np.int64)

    # Safety checks.
    if X.shape[1] != 64:
        print(f"WARNING: {subject} has {X.shape[1]} EEG channels; expected 64 for standard EEGMMIDB.")
    if X.shape[2] < 100:
        raise RuntimeError(f"Unexpected epoch length for {subject}: {X.shape}")

    np.savez_compressed(cache_path, X=X, y=y)
    return X, y

# Cache all available subjects.
cache_summary = []
for s in tqdm(subjects, desc="Caching subjects"):
    Xs, ys = preprocess_subject(s, subjects_map[s], force_recompute=False)
    cache_summary.append({
        "subject": s,
        "epochs": len(ys),
        "channels": Xs.shape[1],
        "samples": Xs.shape[2],
        "left": int((ys == 0).sum()),
        "right": int((ys == 1).sum()),
        "feet": int((ys == 2).sum()),
    })
    del Xs, ys
    gc.collect()

cache_df = pd.DataFrame(cache_summary)
print(cache_df.head())
print(cache_df[["epochs", "left", "right", "feet"]].sum())

Caching subjects: 0it [00:00, ?it/s]

Empty DataFrame
Columns: []
Index: []


KeyError: "None of [Index(['epochs', 'left', 'right', 'feet'], dtype='object')] are in the [columns]"

In [ ]:
# CELL 8 — Inspect class balance and an example subject

print(cache_df.describe(include="all"))
print("\nMean epochs/subject:", cache_df.epochs.mean())
print("\nClass totals:")
print(cache_df[["left", "right", "feet"]].sum())

# Plot one representative channel / epoch for sanity checking.
s0 = subjects[0]
d0 = np.load(Path(cfg.CACHE_DIR) / f"{s0}_3class.npz")
X0, y0 = d0["X"], d0["y"]

idx = int(np.where(y0 == 0)[0][0])
t = np.arange(X0.shape[-1]) / cfg.SFREQ

plt.figure(figsize=(12, 3))
plt.plot(t, X0[idx, 0])
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title(f"{s0} — one Left Hand epoch — channel 0")
plt.tight_layout()
plt.show()
del d0, X0, y0

In [ ]:
# CELL 9 — Leakage-safe normalization and fold assembly

def fit_channel_normalizer(train_subjects: List[str]):
    total_n = 0
    sum_c = None
    sumsq_c = None

    for s in train_subjects:
        d = np.load(Path(cfg.CACHE_DIR) / f"{s}_3class.npz", mmap_mode="r")
        X = d["X"]
        # Sum over trials and time, retaining channel dimension.
        xc = X.transpose(1, 0, 2).reshape(X.shape[1], -1).astype(np.float64)
        if sum_c is None:
            sum_c = np.zeros(xc.shape[0], dtype=np.float64)
            sumsq_c = np.zeros(xc.shape[0], dtype=np.float64)
        sum_c += xc.sum(axis=1)
        sumsq_c += np.square(xc).sum(axis=1)
        total_n += xc.shape[1]
        del d

    mean = sum_c / total_n
    var = np.maximum(sumsq_c / total_n - mean ** 2, 1e-10)
    std = np.sqrt(var)
    return mean.astype(np.float32), std.astype(np.float32)


def apply_normalizer(X, mean, std):
    return ((X - mean[None, :, None]) / std[None, :, None]).astype(np.float32)


def load_subjects(subject_list, mean=None, std=None):
    Xs, ys, gs = [], [], []
    for s in subject_list:
        d = np.load(Path(cfg.CACHE_DIR) / f"{s}_3class.npz")
        X = d["X"].astype(np.float32)
        y = d["y"].astype(np.int64)
        if mean is not None:
            X = apply_normalizer(X, mean, std)
        Xs.append(X)
        ys.append(y)
        gs.append(np.full(len(y), s))
    X = np.concatenate(Xs, axis=0)
    y = np.concatenate(ys, axis=0)
    g = np.concatenate(gs, axis=0)
    return X, y, g

In [ ]:
# CELL 10 — Dataset augmentation utilities

def augment_batch(x: torch.Tensor, training=True) -> torch.Tensor:
    """x: [B,C,T]. Augmentation is label-preserving for motor imagery windows."""
    if not training:
        return x

    b, c, t = x.shape
    out = x.clone()

    # Amplitude scaling.
    scale = torch.empty(b, 1, 1, device=x.device).uniform_(cfg.AMP_SCALE_MIN, cfg.AMP_SCALE_MAX)
    out = out * scale

    # Small Gaussian noise.
    out = out + cfg.NOISE_STD * torch.randn_like(out)

    # Random channel dropout.
    if cfg.CHANNEL_DROP_P > 0:
        keep = (torch.rand(b, c, 1, device=x.device) > cfg.CHANNEL_DROP_P).float()
        out = out * keep

    # Time masking.
    if cfg.TIME_MASK_P > 0:
        for i in range(b):
            if random.random() < cfg.TIME_MASK_P:
                max_len = max(1, int(t * cfg.TIME_MASK_MAX_FRAC))
                m = random.randint(1, max_len)
                start = random.randint(0, max(0, t - m))
                out[i, :, start:start+m] = 0.0

    # Small circular temporal shift.
    if cfg.MAX_SHIFT > 0:
        shifts = torch.randint(-cfg.MAX_SHIFT, cfg.MAX_SHIFT + 1, (b,), device=x.device)
        out = torch.stack([torch.roll(out[i], int(shifts[i].item()), dims=-1) for i in range(b)])

    return out

In [ ]:
# CELL 11 — Model components

class GradReverseFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None


def grad_reverse(x, lambda_=1.0):
    return GradReverseFn.apply(x, lambda_)


class SE1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(4, channels // reduction)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden), nn.GELU(),
            nn.Linear(hidden, channels), nn.Sigmoid()
        )
    def forward(self, x):
        z = self.pool(x).squeeze(-1)
        a = self.fc(z).unsqueeze(-1)
        return x * a


class ResidualTCNBlock(nn.Module):
    def __init__(self, channels, kernel=5, dilation=1, dropout=0.15):
        super().__init__()
        pad = ((kernel - 1) * dilation) // 2
        self.net = nn.Sequential(
            nn.Conv1d(channels, channels, kernel, padding=pad, dilation=dilation, groups=channels, bias=False),
            nn.BatchNorm1d(channels), nn.GELU(),
            nn.Conv1d(channels, channels, 1, bias=False),
            nn.BatchNorm1d(channels), nn.GELU(),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        return x + self.net(x)


class LearnableFilterBank(nn.Module):
    def __init__(self, channels, bands, kernel_size=63, fs=160.0):
        super().__init__()
        self.channels = channels
        self.bands = bands
        self.kernel_size = kernel_size
        self.conv = nn.Conv1d(
            channels, channels * len(bands),
            kernel_size=kernel_size, padding=kernel_size//2,
            groups=channels, bias=False
        )
        self._init_fir(fs)

    def _init_fir(self, fs):
        # Initialize each depthwise filter with a classical FIR band-pass.
        k = self.kernel_size
        weights = []
        for lo, hi in self.bands:
            h = signal.firwin(k, [lo, hi], pass_zero=False, fs=fs)
            weights.append(h.astype(np.float32))
        w = np.stack(weights, axis=0)  # [BANDS,K]
        w = np.tile(w[:, None, :], (self.channels, 1, 1))
        # Conv layout for groups=channels: [channels*bands, 1, K]
        self.conv.weight.data.copy_(torch.tensor(w).reshape(self.channels * len(self.bands), 1, k))

    def forward(self, x):
        return self.conv(x)


class MultiScaleTemporal(nn.Module):
    def __init__(self, in_ch, branch_ch, dropout=0.15):
        super().__init__()
        kernels = [3, 7, 15, 31]
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(in_ch, branch_ch, k, padding=k//2, bias=False),
                nn.BatchNorm1d(branch_ch), nn.GELU(),
                nn.Dropout(dropout)
            ) for k in kernels
        ])
        self.out_ch = branch_ch * len(kernels)
        self.mix = nn.Sequential(
            nn.Conv1d(self.out_ch, self.out_ch, 1, bias=False),
            nn.BatchNorm1d(self.out_ch), nn.GELU()
        )

    def forward(self, x):
        return self.mix(torch.cat([b(x) for b in self.branches], dim=1))


class AttentionPool(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(dim, dim//2), nn.Tanh(), nn.Linear(dim//2, 1))
    def forward(self, x):
        # x: [B,T,D]
        a = torch.softmax(self.score(x).squeeze(-1), dim=1)
        return torch.sum(x * a.unsqueeze(-1), dim=1)


class USDA_Net(nn.Module):
    def __init__(self, input_channels, n_classes, n_domains, cfg):
        super().__init__()
        self.input_channels = input_channels
        self.n_classes = n_classes
        self.n_domains = n_domains

        # Dataset-specific 1x1 projection -> fixed internal montage size.
        self.input_proj = nn.Sequential(
            nn.Conv1d(input_channels, cfg.MODEL_CHANNELS, 1, bias=False),
            nn.BatchNorm1d(cfg.MODEL_CHANNELS), nn.GELU()
        )

        self.temporal = MultiScaleTemporal(
            cfg.MODEL_CHANNELS, cfg.TEMP_BRANCH_CHANNELS, cfg.DROPOUT
        )

        self.filterbank = LearnableFilterBank(
            cfg.MODEL_CHANNELS, cfg.SPECTRAL_BANDS, cfg.FILTER_LEN, cfg.SFREQ
        )
        spec_in = cfg.MODEL_CHANNELS * len(cfg.SPECTRAL_BANDS)
        self.spectral = nn.Sequential(
            nn.Conv1d(spec_in, self.temporal.out_ch, 1, bias=False),
            nn.BatchNorm1d(self.temporal.out_ch), nn.GELU()
        )

        fused_ch = self.temporal.out_ch * 2
        self.fuse = nn.Sequential(
            nn.Conv1d(fused_ch, cfg.TOKEN_DIM, 1, bias=False),
            nn.BatchNorm1d(cfg.TOKEN_DIM), nn.GELU(),
            SE1D(cfg.TOKEN_DIM),
            ResidualTCNBlock(cfg.TOKEN_DIM, kernel=5, dilation=1, dropout=cfg.DROPOUT),
            ResidualTCNBlock(cfg.TOKEN_DIM, kernel=7, dilation=2, dropout=cfg.DROPOUT),
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=cfg.TOKEN_DIM,
            nhead=cfg.N_HEADS,
            dim_feedforward=cfg.FF_DIM,
            dropout=cfg.DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=cfg.N_TRANSFORMER_LAYERS)
        self.pool = AttentionPool(cfg.TOKEN_DIM)

        self.embedding = nn.Sequential(
            nn.Linear(cfg.TOKEN_DIM, cfg.EMBED_DIM),
            nn.LayerNorm(cfg.EMBED_DIM), nn.GELU(), nn.Dropout(cfg.DROPOUT)
        )

        self.classifier = nn.Linear(cfg.EMBED_DIM, n_classes)
        self.domain_classifier = nn.Sequential(
            nn.Linear(cfg.EMBED_DIM, cfg.DOMAIN_HIDDEN), nn.GELU(), nn.Dropout(cfg.DROPOUT),
            nn.Linear(cfg.DOMAIN_HIDDEN, n_domains)
        )

    def encode(self, x):
        x = self.input_proj(x)
        ft = self.temporal(x)
        fs = self.spectral(self.filterbank(x))
        z = self.fuse(torch.cat([ft, fs], dim=1))
        # Reduce token count without throwing away the sequence entirely.
        z = F.avg_pool1d(z, kernel_size=8, stride=8)
        z = z.transpose(1, 2)  # [B,T,D]
        z = self.transformer(z)
        z = self.pool(z)
        return self.embedding(z)

    def forward(self, x, domain_lambda=0.0, return_embedding=False):
        emb = self.encode(x)
        cls = self.classifier(emb)
        dom = self.domain_classifier(grad_reverse(emb, domain_lambda))
        if return_embedding:
            return cls, dom, emb
        return cls, dom

In [ ]:
# CELL 12 — Prototype bank + supervised contrastive loss

class PrototypeBank:
    def __init__(self, n_classes, embed_dim, momentum=0.90, device=DEVICE):
        self.n_classes = n_classes
        self.embed_dim = embed_dim
        self.momentum = momentum
        self.prototypes = torch.zeros(n_classes, embed_dim, device=device)
        self.initialized = torch.zeros(n_classes, dtype=torch.bool, device=device)

    @torch.no_grad()
    def update(self, z, y):
        z = F.normalize(z.detach(), dim=1)
        for c in range(self.n_classes):
            mask = (y == c)
            if mask.any():
                mean = F.normalize(z[mask].mean(dim=0, keepdim=True), dim=1).squeeze(0)
                if self.initialized[c]:
                    self.prototypes[c] = F.normalize(
                        self.momentum * self.prototypes[c] + (1-self.momentum) * mean, dim=0
                    )
                else:
                    self.prototypes[c] = mean
                    self.initialized[c] = True


def prototype_loss(z, y, bank: PrototypeBank):
    z = F.normalize(z, dim=1)
    valid = bank.initialized[y]
    if not valid.any():
        return torch.zeros((), device=z.device)
    sims = (z[valid] * bank.prototypes[y[valid]]).sum(dim=1)
    return (1.0 - sims).mean()


def supervised_contrastive_loss(features, labels, temperature=0.1):
    # features: [B,2,D]
    device = features.device
    b, n_views, d = features.shape
    feats = F.normalize(features, dim=-1)
    contrast = torch.cat(torch.unbind(feats, dim=1), dim=0)  # [B*2,D]
    logits = contrast @ contrast.T / temperature

    logits_mask = ~torch.eye(b * n_views, dtype=torch.bool, device=device)
    logits = logits.masked_fill(~logits_mask, -1e9)

    labels = labels.contiguous().view(-1, 1)
    labels_all = labels.repeat(n_views, 1)
    mask = torch.eq(labels_all, labels_all.T) & logits_mask

    log_prob = logits - torch.logsumexp(logits, dim=1, keepdim=True)
    positives = mask.sum(dim=1)
    valid = positives > 0
    if not valid.any():
        return torch.zeros((), device=device)
    mean_log_prob_pos = (mask * log_prob).sum(dim=1) / positives.clamp_min(1)
    return -mean_log_prob_pos[valid].mean()

In [ ]:
# CELL 13 — Training / validation helpers

def domain_lambda(epoch, total_epochs, warmup_epochs):
    if epoch < warmup_epochs:
        return 0.0
    p = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs - 1)
    p = min(max(p, 0.0), 1.0)
    return float((2.0 / (1.0 + math.exp(-10 * p)) - 1.0) * cfg.LAMBDA_DOMAIN_MAX)


def make_loader(X, y, train=True):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    if not train:
        return DataLoader(ds, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=cfg.NUM_WORKERS, pin_memory=True)

    # Balanced batch sampling protects against small subject/run count differences.
    class_counts = np.bincount(y, minlength=cfg.N_CLASSES).astype(np.float64)
    class_weights = 1.0 / np.maximum(class_counts, 1.0)
    sample_weights = class_weights[y]
    sampler = WeightedRandomSampler(
        torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(y), replacement=True
    )
    return DataLoader(ds, batch_size=cfg.BATCH_SIZE, sampler=sampler,
                      num_workers=cfg.NUM_WORKERS, pin_memory=True)


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "kappa": cohen_kappa_score(y_true, y_pred),
    }

@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    all_y, all_pred, all_prob = [], [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        logits, _ = model(x, domain_lambda=0.0)
        prob = torch.softmax(logits, dim=1)
        pred = prob.argmax(dim=1)
        all_y.append(y.numpy())
        all_pred.append(pred.cpu().numpy())
        all_prob.append(prob.cpu().numpy())
    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_pred)
    prob = np.concatenate(all_prob)
    return compute_metrics(y_true, y_pred), y_true, y_pred, prob

In [ ]:
# CELL 14 — Single-fold training function

def train_one_fold(train_subjects, val_subjects, test_subject, fold_id=0, return_model=False):
    print(f"\n{'='*90}")
    print(f"FOLD {fold_id} | TEST SUBJECT = {test_subject}")
    print(f"Train subjects: {len(train_subjects)} | Val subjects: {len(val_subjects)}")
    print(f"{'='*90}")

    # Fit normalization ONLY on the inner-training subjects.
    # Validation and the held-out test subject are strictly transform-only.
    mean, std = fit_channel_normalizer(train_subjects)

    X_train, y_train, _ = load_subjects(train_subjects, mean, std)
    X_val, y_val, _ = load_subjects(val_subjects, mean, std)
    X_test, y_test, _ = load_subjects([test_subject], mean, std)

    n_channels = X_train.shape[1]
    n_domains = len(train_subjects)
    domain_to_idx = {s:i for i,s in enumerate(train_subjects)}

    # Domain IDs per training trial.
    _, _, train_groups = load_subjects(train_subjects, mean=None, std=None)
    train_domain = np.asarray([domain_to_idx[s] for s in train_groups], dtype=np.int64)

    train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train), torch.from_numpy(train_domain))
    class_counts = np.bincount(y_train, minlength=cfg.N_CLASSES).astype(np.float64)
    sample_weights = (1.0 / np.maximum(class_counts, 1.0))[y_train]
    sampler = WeightedRandomSampler(torch.as_tensor(sample_weights, dtype=torch.double), len(y_train), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, sampler=sampler,
                              num_workers=cfg.NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = make_loader(X_val, y_val, train=False)
    test_loader = make_loader(X_test, y_test, train=False)

    model = USDA_Net(n_channels, cfg.N_CLASSES, n_domains, cfg).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS)
    ce = nn.CrossEntropyLoss(label_smoothing=cfg.LABEL_SMOOTHING)
    prototypes = PrototypeBank(cfg.N_CLASSES, cfg.EMBED_DIM, momentum=0.90, device=DEVICE)

    best_score = -np.inf
    best_state = None
    patience_count = 0
    history = []

    use_amp = DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp) if hasattr(torch, "amp") else None

    for epoch in range(cfg.EPOCHS):
        model.train()
        dlambda = domain_lambda(epoch, cfg.EPOCHS, cfg.WARMUP_EPOCHS)
        running = {k: 0.0 for k in ["total", "cls", "domain", "proto", "supcon"]}
        n_seen = 0

        pbar = tqdm(train_loader, desc=f"Fold {fold_id} Ep {epoch+1:03d}/{cfg.EPOCHS}", leave=False)
        for x, y, d in pbar:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            d = d.to(DEVICE, non_blocking=True)

            x1 = augment_batch(x, training=True)
            x2 = augment_batch(x, training=True)

            optimizer.zero_grad(set_to_none=True)
            if use_amp:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    logits1, dom1, z1 = model(x1, domain_lambda=dlambda, return_embedding=True)
                    _, _, z2 = model(x2, domain_lambda=0.0, return_embedding=True)
                    l_cls = ce(logits1, y)
                    l_dom = F.cross_entropy(dom1, d)
                    l_proto = prototype_loss(z1, y, prototypes)
                    l_supcon = supervised_contrastive_loss(torch.stack([z1, z2], dim=1), y, cfg.SUPCON_TEMP)
                    loss = l_cls + dlambda * l_dom + cfg.LAMBDA_PROTO * l_proto + cfg.LAMBDA_SUPCON * l_supcon
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), cfg.CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()
            else:
                logits1, dom1, z1 = model(x1, domain_lambda=dlambda, return_embedding=True)
                _, _, z2 = model(x2, domain_lambda=0.0, return_embedding=True)
                l_cls = ce(logits1, y)
                l_dom = F.cross_entropy(dom1, d)
                l_proto = prototype_loss(z1, y, prototypes)
                l_supcon = supervised_contrastive_loss(torch.stack([z1, z2], dim=1), y, cfg.SUPCON_TEMP)
                loss = l_cls + dlambda * l_dom + cfg.LAMBDA_PROTO * l_proto + cfg.LAMBDA_SUPCON * l_supcon
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), cfg.CLIP_NORM)
                optimizer.step()

            prototypes.update(z1, y)
            bs = len(y)
            n_seen += bs
            running["total"] += float(loss.detach().cpu()) * bs
            running["cls"] += float(l_cls.detach().cpu()) * bs
            running["domain"] += float(l_dom.detach().cpu()) * bs
            running["proto"] += float(l_proto.detach().cpu()) * bs
            running["supcon"] += float(l_supcon.detach().cpu()) * bs
            pbar.set_postfix(loss=f"{running['total']/n_seen:.3f}", dom=f"{dlambda:.2f}")

        scheduler.step()
        train_stats = {k:v/max(1,n_seen) for k,v in running.items()}
        val_metrics, _, _, _ = evaluate_model(model, val_loader)
        row = {"epoch":epoch+1, "domain_lambda":dlambda, **train_stats, **{f"val_{k}":v for k,v in val_metrics.items()}}
        history.append(row)

        score = val_metrics["balanced_accuracy"]
        print(
            f"Epoch {epoch+1:03d} | "
            f"loss={train_stats['total']:.4f} | cls={train_stats['cls']:.4f} | "
            f"dom={train_stats['domain']:.4f} | λ={dlambda:.3f} | "
            f"val_acc={val_metrics['accuracy']*100:.2f}% | "
            f"val_bacc={val_metrics['balanced_accuracy']*100:.2f}% | "
            f"val_f1={val_metrics['macro_f1']*100:.2f}%"
        )

        if score > best_score:
            best_score = score
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= cfg.PATIENCE:
                print("Early stopping.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    test_metrics, yt, yp, probs = evaluate_model(model, test_loader)
    cm = confusion_matrix(yt, yp, labels=list(range(cfg.N_CLASSES)))

    print("\nTEST RESULTS")
    print(json.dumps(test_metrics, indent=2))
    print("Confusion matrix:\n", cm)

    if cfg.SAVE_CHECKPOINTS:
        ckpt = {
            "fold_id": fold_id,
            "test_subject": test_subject,
            "train_subjects": train_subjects,
            "val_subjects": val_subjects,
            "mean": mean,
            "std": std,
            "config": asdict(cfg),
            "model_state": model.state_dict(),
        }
        torch.save(ckpt, Path(cfg.OUTPUT_DIR) / f"fold_{fold_id:03d}_{test_subject}.pt")

    history_df = pd.DataFrame(history)
    history_df.to_csv(Path(cfg.OUTPUT_DIR) / f"history_fold_{fold_id:03d}_{test_subject}.csv", index=False)

    out = {
        "fold_id": fold_id,
        "test_subject": test_subject,
        **test_metrics,
        "n_test": len(yt),
        "confusion_matrix": cm.tolist(),
        "history": history_df,
        "y_true": yt,
        "y_pred": yp,
        "probs": probs,
        "model": model if return_model else None,
        "mean": mean,
        "std": std,
    }

    # Explicitly free large fold arrays when model is not requested.
    del X_train, y_train, X_val, y_val, X_test, y_test
    del train_loader, val_loader, test_loader
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return out

In [ ]:
# CELL 15 — Define the LOSO fold order and select development/full mode

if cfg.RUN_FULL_LOSO:
    selected_subjects = subjects.copy()
elif cfg.MAX_FOLDS is not None:
    selected_subjects = subjects[:cfg.MAX_FOLDS]
else:
    selected_subjects = subjects[:cfg.QUICK_FOLDS] if cfg.RUN_QUICK_DEMO else subjects.copy()

print("Selected outer folds:", len(selected_subjects))
print(selected_subjects)

In [ ]:
# CELL 16 — Run strict subject-independent LOSO

# IMPORTANT:
# Every outer fold holds out the test subject completely.
# Inner validation subjects are sampled only from the remaining source subjects.
# The test subject is never used for model selection, normalization fitting, or training.

loso_results = []
all_confusions = []
fold_outputs = []

for fold_id, test_subject in enumerate(selected_subjects, start=1):
    source_subjects = [s for s in subjects if s != test_subject]
    gss = GroupShuffleSplit(
        n_splits=1,
        test_size=max(cfg.VAL_SUBJECT_FRAC, 1.0 / max(len(source_subjects), 2)),
        random_state=SEED + fold_id
    )

    # We split the subject list itself so entire subjects belong to either train or validation.
    idx = np.arange(len(source_subjects))
    train_idx, val_idx = next(gss.split(idx, groups=np.array(source_subjects)))
    train_subjects = [source_subjects[i] for i in train_idx]
    val_subjects = [source_subjects[i] for i in val_idx]

    out = train_one_fold(train_subjects, val_subjects, test_subject, fold_id=fold_id, return_model=False)
    fold_outputs.append(out)

    row = {
        "fold": fold_id,
        "subject": test_subject,
        "accuracy": out["accuracy"],
        "balanced_accuracy": out["balanced_accuracy"],
        "macro_f1": out["macro_f1"],
        "kappa": out["kappa"],
        "n_test": out["n_test"],
    }
    loso_results.append(row)
    all_confusions.append(np.asarray(out["confusion_matrix"]))

    pd.DataFrame(loso_results).to_csv(Path(cfg.OUTPUT_DIR) / "loso_results_running.csv", index=False)

loso_df = pd.DataFrame(loso_results)
print("\nLOSO results so far:")
print(loso_df)

In [ ]:
# CELL 17 — Publication-style LOSO summary and confidence intervals

def mean_ci(values, confidence=0.95):
    values = np.asarray(values, dtype=float)
    n = len(values)
    mean = values.mean()
    if n < 2:
        return mean, np.nan, np.nan
    se = stats.sem(values)
    h = se * stats.t.ppf((1 + confidence) / 2.0, n - 1)
    return mean, mean - h, mean + h

summary_rows = []
for metric in ["accuracy", "balanced_accuracy", "macro_f1", "kappa"]:
    mean, lo, hi = mean_ci(loso_df[metric].values)
    summary_rows.append({
        "metric": metric,
        "mean": mean,
        "std": loso_df[metric].std(ddof=1) if len(loso_df) > 1 else np.nan,
        "95ci_low": lo,
        "95ci_high": hi,
    })
summary_df = pd.DataFrame(summary_rows)
print(summary_df)

print("\nPrimary metric — mean LOSO balanced accuracy:",
      f"{loso_df['balanced_accuracy'].mean()*100:.2f}%")
print("Primary metric — mean LOSO accuracy:",
      f"{loso_df['accuracy'].mean()*100:.2f}%")
print("80% target gap (accuracy):",
      f"{(0.80 - loso_df['accuracy'].mean())*100:+.2f} percentage points")

loso_df.to_csv(Path(cfg.OUTPUT_DIR) / "loso_results_final.csv", index=False)
summary_df.to_csv(Path(cfg.OUTPUT_DIR) / "loso_summary.csv", index=False)

In [ ]:
# CELL 18 — Aggregate confusion matrix and per-subject accuracy plot

if all_confusions:
    cm_total = np.sum(np.stack(all_confusions), axis=0)
else:
    cm_total = np.zeros((cfg.N_CLASSES, cfg.N_CLASSES), dtype=int)

plt.figure(figsize=(5, 4))
plt.imshow(cm_total, interpolation="nearest")
plt.title("Aggregated LOSO Confusion Matrix")
plt.colorbar()
plt.xticks(range(cfg.N_CLASSES), cfg.CLASS_NAMES, rotation=20, ha="right")
plt.yticks(range(cfg.N_CLASSES), cfg.CLASS_NAMES)
for i in range(cfg.N_CLASSES):
    for j in range(cfg.N_CLASSES):
        plt.text(j, i, str(cm_total[i, j]), ha="center", va="center")
plt.ylabel("True")
plt.xlabel("Predicted")
plt.tight_layout()
plt.savefig(Path(cfg.OUTPUT_DIR) / "loso_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

if len(loso_df):
    plt.figure(figsize=(12, 4))
    plt.plot(loso_df["subject"], loso_df["accuracy"] * 100, marker="o", linewidth=1.5, label="Accuracy")
    plt.axhline(80.0, linestyle="--", linewidth=1.2, label="80% target")
    plt.xticks(rotation=90)
    plt.ylabel("Accuracy (%)")
    plt.xlabel("Held-out subject")
    plt.title("Strict LOSO Accuracy by Subject")
    plt.legend()
    plt.tight_layout()
    plt.savefig(Path(cfg.OUTPUT_DIR) / "per_subject_accuracy.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# CELL 19 — Optional embedding visualization on the last available fold

# This is diagnostic only. It does NOT affect training or evaluation.
from sklearn.manifold import TSNE

if fold_outputs and fold_outputs[-1].get("model") is not None:
    # This path is normally disabled because return_model=False in the LOSO loop.
    print("No diagnostic model retained by default.")
else:
    print("Embedding visualization is optional; rerun one fold with return_model=True to enable it.")

In [ ]:
# CELL 20 — Optional: run one diagnostic fold and visualize learned embeddings

RUN_EMBEDDING_DIAGNOSTIC = False

if RUN_EMBEDDING_DIAGNOSTIC:
    test_subject = selected_subjects[0]
    source_subjects = [s for s in subjects if s != test_subject]
    gss = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=SEED)
    idx = np.arange(len(source_subjects))
    tr, va = next(gss.split(idx, groups=np.array(source_subjects)))
    out = train_one_fold(
        [source_subjects[i] for i in tr],
        [source_subjects[i] for i in va],
        test_subject,
        fold_id=999,
        return_model=True
    )
    model = out["model"]
    mean, std = out["mean"], out["std"]
    Xtest, ytest, _ = load_subjects([test_subject], mean, std)
    loader = make_loader(Xtest, ytest, train=False)
    embs, ys = [], []
    with torch.no_grad():
        model.eval()
        for x, y in loader:
            z = model.encode(x.to(DEVICE)).cpu().numpy()
            embs.append(z)
            ys.append(y.numpy())
    embs = np.concatenate(embs)
    ys = np.concatenate(ys)
    n = min(1500, len(embs))
    take = np.random.default_rng(SEED).choice(len(embs), n, replace=False)
    z2 = TSNE(n_components=2, random_state=SEED, init="pca", learning_rate="auto").fit_transform(embs[take])
    plt.figure(figsize=(7, 6))
    for c, name in enumerate(cfg.CLASS_NAMES):
        m = ys[take] == c
        plt.scatter(z2[m, 0], z2[m, 1], s=10, alpha=0.6, label=name)
    plt.legend()
    plt.title(f"Target-subject embedding — {test_subject}")
    plt.tight_layout()
    plt.show()

## Separate experiment: unlabeled target adaptation

This section must **not** be mixed with strict LOSO numbers.

The adaptation routine uses only unlabeled EEG from the held-out target subject. It does not use target labels. It is therefore a transductive / unsupervised target-adaptation experiment rather than a strict unseen-subject benchmark.

In [ ]:
# CELL 21 — Optional unlabeled target adaptation (separate metric)

def adapt_on_unlabeled_target(model, X_target, steps=5, lr=2e-4, confidence=0.90):
    """
    Unsupervised target adaptation:
      1) teacher predictions are generated from the current model.
      2) confident pseudo-labels are selected.
      3) model is updated with pseudo-label CE + augmentation consistency.
    No target labels are used.
    """
    model = model.to(DEVICE)
    model.train()
    loader = make_loader(X_target, np.zeros(len(X_target), dtype=np.int64), train=False)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)

    for ep in range(steps):
        total_loss = 0.0
        n = 0
        for x, _ in loader:
            x = x.to(DEVICE)
            with torch.no_grad():
                logits, _ = model(x, domain_lambda=0.0)
                p = torch.softmax(logits, dim=1)
                conf, pseudo = p.max(dim=1)
            keep = conf >= confidence
            if keep.sum() < 2:
                continue

            x_aug = augment_batch(x[keep], training=True)
            logits_aug, _ = model(x_aug, domain_lambda=0.0)
            # Prediction consistency / pseudo-label objective.
            loss = F.cross_entropy(logits_aug, pseudo[keep])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total_loss += float(loss.detach().cpu()) * int(keep.sum())
            n += int(keep.sum())
        print(f"Adapt epoch {ep+1}/{steps} | pseudo-labeled samples={n} | loss={total_loss/max(n,1):.4f}")
    model.eval()
    return model

print("Target adaptation function defined. Keep cfg.RUN_TARGET_ADAPTATION=False for the primary strict benchmark.")

In [ ]:
# CELL 22 — Optional target adaptation evaluation for a selected fold

RUN_SELECTED_ADAPTATION = False
ADAPT_FOLD_INDEX = 0

if RUN_SELECTED_ADAPTATION:
    # To keep strict and adapted results separate, retrain one selected fold and adapt only afterwards.
    test_subject = selected_subjects[ADAPT_FOLD_INDEX]
    source_subjects = [s for s in subjects if s != test_subject]
    gss = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=SEED + 123)
    idx = np.arange(len(source_subjects))
    tr, va = next(gss.split(idx, groups=np.array(source_subjects)))
    out = train_one_fold(
        [source_subjects[i] for i in tr],
        [source_subjects[i] for i in va],
        test_subject,
        fold_id=1000 + ADAPT_FOLD_INDEX,
        return_model=True
    )
    model = out["model"]
    Xtarget, ytarget, _ = load_subjects([test_subject], out["mean"], out["std"])
    strict_metrics, _, _, _ = evaluate_model(model, make_loader(Xtarget, ytarget, train=False))
    print("Strict target metrics:", strict_metrics)
    model_adapted = adapt_on_unlabeled_target(
        model, Xtarget,
        steps=cfg.ADAPT_EPOCHS,
        lr=cfg.ADAPT_LR,
        confidence=cfg.PSEUDO_CONFIDENCE,
    )
    adapted_metrics, _, _, _ = evaluate_model(model_adapted, make_loader(Xtarget, ytarget, train=False))
    print("Unlabeled target-adapted metrics:", adapted_metrics)

## Ablation protocol

For a paper, do not report only the full model. Use the exact same LOSO folds and training protocol while switching components off:

- Baseline: CE only
- + Multi-scale spectral/temporal encoder
- + DANN
- + Prototype loss
- + Supervised contrastive loss
- Full USDA-Net

Run ablations on a fixed subset first, then full LOSO only for configurations that are computationally feasible.

In [ ]:
# CELL 23 — Ablation configuration helper

ABLATIONS = {
    "baseline_ce": dict(use_domain=False, use_proto=False, use_supcon=False),
    "dann": dict(use_domain=True, use_proto=False, use_supcon=False),
    "dann_proto": dict(use_domain=True, use_proto=True, use_supcon=False),
    "dann_proto_supcon": dict(use_domain=True, use_proto=True, use_supcon=True),
}

print("Ablation definitions:")
for k,v in ABLATIONS.items():
    print(k, v)
print("Note: the main training function above is the full model. To run the formal ablation table, expose these switches inside CELL 14 and keep the same outer folds.")

In [ ]:
# CELL 24 — Save an experiment manifest

manifest = {
    "dataset": "PhysioNet EEGMMIDB",
    "task": "3-class motor imagery: Left Hand / Right Hand / Feet",
    "protocol": "strict LOSO",
    "selected_subjects": selected_subjects,
    "config": asdict(cfg),
    "device": str(DEVICE),
}
Path(cfg.OUTPUT_DIR, "experiment_manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
print("Saved:", Path(cfg.OUTPUT_DIR, "experiment_manifest.json").resolve())

## Research interpretation

Do not write “80% achieved” until the full LOSO experiment has actually been run and the mean subject-level accuracy is at least 0.80. The notebook reports the exact observed mean, standard deviation, confidence interval, per-subject scores, and confusion matrix.

For the strict benchmark, the key number is **mean accuracy across held-out subjects**. Balanced accuracy, macro-F1, and Cohen's kappa should be reported alongside it.

In [ ]:
# CELL 25 — Final reporting cell

if len(loso_df) > 0:
    report = {
        "n_evaluated_subjects": int(len(loso_df)),
        "mean_accuracy": float(loso_df.accuracy.mean()),
        "std_accuracy": float(loso_df.accuracy.std(ddof=1)) if len(loso_df) > 1 else float("nan"),
        "mean_balanced_accuracy": float(loso_df.balanced_accuracy.mean()),
        "mean_macro_f1": float(loso_df.macro_f1.mean()),
        "mean_kappa": float(loso_df.kappa.mean()),
        "accuracy_target_80pct_met": bool(loso_df.accuracy.mean() >= 0.80),
    }
    print(json.dumps(report, indent=2))
    print("\nSaved outputs in:", Path(cfg.OUTPUT_DIR).resolve())
else:
    print("Run CELL 16 first.")